# Step 1 · Hello, LLM, with Bifrost and Langfuse
In `00_hello_ollama` we called Ollama directly. Now the same model sits behind two more parts:

- **Bifrost**, the gateway: the only door to the models, and it refuses anything not on its list.
- **Langfuse**, self-hosted: a record of every call. It runs on this machine too, so nothing leaves the laptop.

> **Standalone.** All the logic is in the cells below. Nothing is imported from the app.
> Start Bifrost and Langfuse first (SETUP.md, section 5: Bifrost at `http://127.0.0.1:8080/v1`, Langfuse at `http://127.0.0.1:3000`).
> Install what it needs with `pip install -r requirements.txt`.

## 1. Configure Bifrost and Langfuse
**Bifrost:** we point the normal OpenAI client at Bifrost instead of at Ollama. The code asks for `analyst`, `judge` or `embed`, and only the `MODELS` map names a model, so swapping one is a one-line change.

**Langfuse:** `chat()` wraps each call in a Langfuse *generation*, which records the question, the answer and the tokens. The keys come from `copilot/.env.local`, which `create_secrets.sh` made on this machine.

In [ ]:
# UPDATED CELL (not run yet): the same as the cell below, without the step 4 leftovers
# (the "judge" model and the Ragas setting). When you teach this notebook:
# run this cell, delete the old cell below, then run the rest as usual.
# Everything this notebook needs is in this file. It imports nothing from copilot/.
import os
os.environ.update(ANONYMIZED_TELEMETRY="False", LANGSMITH_TRACING="false",
                  LANGCHAIN_TRACING_V2="false", HF_HUB_OFFLINE="1")   # nothing leaves the machine

import time
from dotenv import load_dotenv
from langfuse import Langfuse
from openai import OpenAI

load_dotenv("../copilot/.env.local")                                   # the Langfuse keys for this machine
LANGFUSE_URL = os.getenv("LANGFUSE_HOST", "http://127.0.0.1:3000")     # self-hosted: never a cloud address
langfuse = Langfuse(public_key=os.environ["LANGFUSE_PUBLIC_KEY"],
                secret_key=os.environ["LANGFUSE_SECRET_KEY"], host=LANGFUSE_URL)

GATEWAY_URL = os.getenv("BIFROST_URL", "http://127.0.0.1:8080/v1")     # Bifrost, running on this machine
MODELS = {"analyst": "ollama/qwen2.5:7b-instruct",     # the role -> model map: code asks for a role
          "embed":   "ollama/nomic-embed-text"}
client = OpenAI(base_url=GATEWAY_URL, api_key="local") # a local gateway needs no real key, but the SDK wants a value


class GatewayError(RuntimeError):
    """A model call failed. There is no fallback model: fail loudly."""


def chat(role, prompt, json_mode=False, max_tokens=700):
    """Send one prompt to the model behind a role, and record the call in Langfuse."""
    kwargs = {"response_format": {"type": "json_object"}} if json_mode else {}
    with langfuse.start_as_current_observation(as_type="generation",
                                        name=f"chat: {role}", 
                                    model=MODELS[role], input=prompt) as generation:
        try:
            reply = client.chat.completions.create(model=MODELS[role], temperature=0,
                                        max_tokens=max_tokens, timeout=900,
                                messages=[{"role": "user", "content": prompt}],
                        **kwargs)
        except Exception as exc:
            generation.update(level="ERROR", status_message=str(exc)[:200])
            raise GatewayError(f"Model role '{role}' failed: {exc}") from exc
        text = (reply.choices[0].message.content or "").strip()
        generation.update(output=text, usage_details={"input": reply.usage.prompt_tokens, "output": reply.usage.completion_tokens})
        return text


def embed(texts):
    """Turn texts into vectors with the embed role."""
    try:
        return [d.embedding for d in client.embeddings.create(model=MODELS["embed"], input=texts).data]
    except Exception as exc:
        raise GatewayError(f"Model role 'embed' failed: {exc}") from exc


print("Bifrost :", GATEWAY_URL)
print("Langfuse:", LANGFUSE_URL, "connected" if langfuse.auth_check() else "NOT connected")

In [ ]:
# Everything this notebook needs is in this file. It imports nothing from copilot/.
import os
os.environ.update(ANONYMIZED_TELEMETRY="False", RAGAS_DO_NOT_TRACK="true", LANGSMITH_TRACING="false",
                  LANGCHAIN_TRACING_V2="false", HF_HUB_OFFLINE="1")   # nothing leaves the machine

import time
from dotenv import load_dotenv
from langfuse import Langfuse
from openai import OpenAI

load_dotenv("../copilot/.env.local")                                   # the Langfuse keys for this machine
LANGFUSE_URL = os.getenv("LANGFUSE_HOST", "http://127.0.0.1:3000")     # self-hosted: never a cloud address
langfuse = Langfuse(public_key=os.environ["LANGFUSE_PUBLIC_KEY"],
                secret_key=os.environ["LANGFUSE_SECRET_KEY"], host=LANGFUSE_URL)

GATEWAY_URL = os.getenv("BIFROST_URL", "http://127.0.0.1:8080/v1")     # Bifrost, running on this machine
MODELS = {"analyst": "ollama/qwen2.5:7b-instruct",     # the role -> model map: code asks for a role
          "judge":   "ollama/llama3.1:8b",
          "embed":   "ollama/nomic-embed-text"}
client = OpenAI(base_url=GATEWAY_URL, api_key="local") # a local gateway needs no real key, but the SDK wants a value


class GatewayError(RuntimeError):
    """A model call failed. There is no fallback model: fail loudly."""


def chat(role, prompt, json_mode=False, max_tokens=700):
    """Send one prompt to the model behind a role, and record the call in Langfuse."""
    kwargs = {"response_format": {"type": "json_object"}} if json_mode else {}
    with langfuse.start_as_current_observation(as_type="generation",
                                        name=f"chat: {role}", 
                                    model=MODELS[role], input=prompt) as generation:
        try:
            reply = client.chat.completions.create(model=MODELS[role], temperature=0,
                                        max_tokens=max_tokens, timeout=900,
                                messages=[{"role": "user", "content": prompt}],
                        **kwargs)
        except Exception as exc:
            generation.update(level="ERROR", status_message=str(exc)[:200])
            raise GatewayError(f"Model role '{role}' failed: {exc}") from exc
        text = (reply.choices[0].message.content or "").strip()
        generation.update(output=text, usage_details={"input": reply.usage.prompt_tokens, "output": reply.usage.completion_tokens})
        return text


def embed(texts):
    """Turn texts into vectors with the embed role."""
    try:
        return [d.embedding for d in client.embeddings.create(model=MODELS["embed"], input=texts).data]
    except Exception as exc:
        raise GatewayError(f"Model role 'embed' failed: {exc}") from exc


print("Bifrost :", GATEWAY_URL)
print("Langfuse:", LANGFUSE_URL, "connected" if langfuse.auth_check() else "NOT connected")

Bifrost : http://127.0.0.1:8080/v1
Langfuse: http://127.0.0.1:3000 connected


## 2. Send one prompt through Bifrost, recorded by Langfuse
The `hello-llm` span groups everything inside it into one trace, and we keep its id to look it up later.

In [2]:
start = time.perf_counter()
with langfuse.start_as_current_observation(as_type="span", name="hello-llm") as span:
    reply = chat("analyst", "In one sentence: what does a payroll provider do for its clients?")
    span.update(output=reply)
    trace_id = span.trace_id
print(reply)
print(f"\n{time.perf_counter() - start:.1f} seconds (the first call also loads the model into memory)")

A payroll provider manages the calculation, processing, and distribution of employees' wages, including deductions and tax withholdings, for its clients.

2.7 seconds (the first call also loads the model into memory)


## 3. How fast is it on this machine?
The same request again, now the model is loaded, reading the token counts the gateway reports.

In [3]:
PROMPT = "In three sentences, explain what a limitation of liability clause does in a contract."
for run in range(1, 4):
    start = time.perf_counter()
    response = client.chat.completions.create(model=MODELS["analyst"], temperature=0, max_tokens=120,
                                              messages=[{"role": "user", "content": PROMPT}])
    seconds = time.perf_counter() - start
    tokens = response.usage.completion_tokens
    print(f"run {run}: {seconds:5.1f}s  {tokens} tokens  {tokens / seconds:4.1f} tokens/second")

run 1:   3.4s  65 tokens  19.1 tokens/second


run 2:   3.6s  77 tokens  21.5 tokens/second


run 3:   3.5s  77 tokens  21.9 tokens/second


## 4. Embeddings: text becomes numbers
Similar meaning gives similar numbers. This is what makes searching a contract by meaning possible.

In [4]:
import math

def cosine(a, b):
    return sum(x * y for x, y in zip(a, b)) / (math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b)))

start = time.perf_counter()
clause, close, far = embed(["The provider's liability is capped at twelve months of fees.",
                            "How much do we pay at most if something goes wrong?",
                            "The cafeteria serves lunch from noon."])
print(f"{len(clause)} numbers per text, 3 texts in {time.perf_counter() - start:.2f}s")
print(f"clause vs related question : {cosine(clause, close):.3f}")
print(f"clause vs unrelated text   : {cosine(clause, far):.3f}")

768 numbers per text, 3 texts in 4.13s
clause vs related question : 0.538
clause vs unrelated text   : 0.348


## 5. The gateway refuses anything not on its allowlist
Bifrost only serves the local models on its allowlist. A different local model and a cloud model are both refused, and the code fails loudly instead of quietly using something else.

In [5]:
for role, model in (("not on the allowlist", "ollama/llama3.2:1b"), ("a cloud model", "openai/gpt-4o")):
    MODELS["test"] = model
    try:
        chat("test", "hi")
        print(f"{role}: ANSWERED (this should not happen)")
    except GatewayError as exc:
        print(f"{role}: refused\n   {str(exc)[:150]}")
del MODELS["test"]

not on the allowlist: refused
   Model role 'test' failed: Error code: 400 - {'type': 'no_key_supports_model', 'is_bifrost_error': False, 'status_code': 400, 'error': {'error': 'no ke
a cloud model: refused
   Model role 'test' failed: Error code: 400 - {'is_bifrost_error': False, 'error': {'error': 'failed to get config for provider openai: not found', 'mes


## 6. Look it up in Langfuse
This reads back what Langfuse saved for the call in section 2. Open the link to see the same thing on screen (logins are in `credentials.md`).

In [6]:
langfuse.flush()      # send anything still waiting

saved = []
for attempt in range(20):       # Langfuse saves traces in the background, so give it a moment
    saved = langfuse.api.observations.get_many(trace_id=trace_id, fields="core,basic,io,model").data
    if any(o.type == "GENERATION" for o in saved):
        break
    time.sleep(3)

for o in saved:
    print(f"{o.type:<11} {o.name:<15} {o.model or ''}")
    if o.type == "GENERATION":
        print(f"   asked   : {o.input}")
        print(f"   answered: {o.output}")
print("\nOpen it:", langfuse.get_trace_url(trace_id=trace_id))

SPAN        hello-llm       
GENERATION  chat: analyst   ollama/qwen2.5:7b-instruct
   asked   : In one sentence: what does a payroll provider do for its clients?
   answered: A payroll provider manages the calculation, processing, and distribution of employees' wages, including deductions and tax withholdings, for its clients.

Open it: http://127.0.0.1:3000/project/contract-copilot/traces/72f4355f6d7c01d7583449479ee67fc9


**Takeaway:** one door to the models, nothing outside the allowlist can run, and every call is on record on this machine.